In [1]:
import pandas as pd
import sqlite3

In [2]:
data_path = "./olist/" # path of csv files 

In [3]:
conn = sqlite3.connect("olist.db") # will automatically create the database 'olist.db'

In [4]:
def load_csv(table_name, file_name):
    df = pd.read_csv(data_path + file_name)
    df.to_sql(table_name, conn, if_exists = "replace", index = False)
    print(f"{table_name}: {len(df)} rows loaded")

This function automates the process of reading a CSV file and inserting its data into a SQL table. It helps avoid writing repetitive code for each dataset.

---

- I read the CSV file using pandas and store it in a DataFrame because DataFrames provide a tabular structure similar to SQL tables, making further processing and database insertion easier.

- `df.to_sql()` : A pandas method used to write DataFrame data into a SQL database.

- `conn` : this declared earlier. It tells pandas **which database to write to.**

- `if_exists="replace"` : Controls what happens if the table already exists.

| Option    | Meaning                       |
| --------- | ----------------------------- |
| `fail`    | Throw error                   |
| `replace` | Delete old table and recreate |
| `append`  | Add rows to existing table    |

`replace` means **drop existing table; create new table; insert fresh date**.

> I used replace so that if the table already exists, it gets recreated with fresh data, preventing duplicate or outdated records.

- If index=True, this index would be saved as a SQL column.

But usually the index is not meaningful data, so we remove it.

Hence: `index=False`

> I set index=False to prevent pandas from writing the DataFrame index as a separate column in the SQL table.

> I print the table name and number of rows loaded to verify that the data ingestion step completed successfully.

In [5]:
load_csv("customers", "olist_customers_dataset.csv")
load_csv("location", "olist_geolocation_dataset.csv")
load_csv("order_items", "olist_order_items_dataset.csv")
load_csv("payments", "olist_order_payments_dataset.csv")
load_csv("reviews", "olist_order_reviews_dataset.csv")
load_csv("orders", "olist_orders_dataset.csv")
load_csv("product", "olist_products_dataset.csv")
load_csv("seller", "olist_sellers_dataset.csv")
load_csv("product_category", "product_category_name_translation.csv")

customers: 99441 rows loaded
location: 1000163 rows loaded
order_items: 112650 rows loaded
payments: 103886 rows loaded
reviews: 99224 rows loaded
orders: 99441 rows loaded
product: 32951 rows loaded
seller: 3095 rows loaded
product_category: 71 rows loaded


In [6]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,customers
1,location
2,order_items
3,payments
4,reviews
5,orders
6,product
7,seller
8,product_category


This query retrieves all table names from the SQLite database by querying the sqlite_master system table and returns the result as a pandas DataFrame using pd.read_sql().

---

- `pd.read_sql()` : This is a Pandas function used to execute a SQL query and return the result as a DataFrame.

> pd.read_sql() executes a SQL query on a database connection and loads the result directly into a pandas DataFrame for analysis.

- `SELECT name FROM sqlite_master WHERE type = 'table'` # object types including table, index, view, trigger, etc.

This query retrieves all table names in a SQLite database.

> sqlite_master is a system table in SQLite that stores metadata about database objects such as tables, indexes, and views.

- `SELECT name` : This retrieves the names of the database objects.
- `conn` : (declared previously) It tells pandas which database to run the query on.

In [7]:
pd.read_sql('''SELECT (SELECT COUNT(*) FROM customers) AS customers,
                      (SELECT COUNT(*) FROM orders) AS orders,
                      (SELECT COUNT(*) FROM payments) AS payments
                      ''', conn)

,customers,orders,payments
0,99441,99441,103886


In [8]:
pd.read_sql('''SELECT COUNT(DISTINCT customer_id) AS id FROM customers''', conn)

,id
0,99441


In [9]:
conn.close()